# Interactive Generation with Your Pretrained GPT

Use this notebook to load a pretrained (or SFT) GPT checkpoint from this repo and generate text.

- Adjust the checkpoint path and model hyperparameters to match your training run
- Tokenizer is set up with the special tokens used in this project
- Generation is simple and fast; for longer/creative outputs, increase `max_new_tokens`

Tip: If you fine-tuned the model (SFT), you can point `checkpoint_path` to an SFT checkpoint instead.


In [45]:
# Setup: imports and utility paths
import os
import torch
from transformers import AutoTokenizer

import gpt  # uses this repo's GPT implementation


# Prefer GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cpu


In [46]:
# Configuration: set your checkpoint path and model config here
# NOTE: Adjust to match the model you trained.

# Example default paths (edit as needed)
# - Pretraining outputs typically saved by pretrain_gpt.py
# - SFT outputs typically saved by sft_gpt.py
 #= "/shared/0/projects/teaching/eecs595/models/pico-gpt/pretrained-models/"  # <- change me
checkpoint_path = 'models/full/model_epoch_1.pt'

# Model architecture used during training
config = {
    "vocab_size": None,          # will be computed from tokenizer below
    "context_length": 1024,
    "emb_dim": 512,
    "n_heads": 8,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": False,
    "ffn_expansion": 8 / 3,
    "ffn_hidden_dim": None,
}

# Generation defaults (you can tweak later)
max_new_tokens = 128
temperature = 0.8
print(checkpoint_path)

models/full/model_epoch_1.pt


In [47]:
# Tokenizer: use repo's setup to ensure special tokens match training
# This mirrors `gpt.setup_tokenizer()` behavior.

tokenizer = gpt.setup_tokenizer()

# Compute actual vocab size that includes special tokens used during training
special_tokens = ["<|user|>", "<|assistant|>", "<|end|>", "<|system|>", "<|pad|>"]
max_token_id = max(tokenizer.convert_tokens_to_ids(tok) for tok in special_tokens)
actual_vocab_size = max_token_id + 1
print(f"Tokenizer vocab size detected: {actual_vocab_size}")

if config["vocab_size"] is None:
    config["vocab_size"] = actual_vocab_size


Tokenizer vocab size detected: 50262


In [48]:
import torch

ckpt = torch.load(checkpoint_path, map_location="cpu")

# Unwrap common wrappers
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# Strip leading prefixes like "_orig_mod." or "module."
for prefix in ("_orig_mod.", "module."):
    if any(k.startswith(prefix) for k in state_dict):
        state_dict = { (k[len(prefix):] if k.startswith(prefix) else k): v
                       for k, v in state_dict.items() }

# Infer vocab size robustly
emb_key = next((k for k in state_dict if k.endswith("embedding.token_embeddings.weight")), None)
if emb_key is None:
    out_key = next((k for k in state_dict if k.endswith("out_head.weight")), None)
    if out_key is None:
        raise KeyError("No embedding or out_head weight found; keys head: " + str(list(state_dict.keys())[:20]))
    original_vocab_size = state_dict[out_key].shape[0]
else:
    original_vocab_size = state_dict[emb_key].shape[0]

print("original_vocab_size:", original_vocab_size)
# Now compare with your config['vocab_size'] and/or load:
# model.load_state_dict(state_dict, strict=False)

original_vocab_size: 50262


In [24]:
print(list(state_dict.keys())[:50])

['embedding.token_embeddings.weight', 'trf_blocks.0.self_attn.mask', 'trf_blocks.0.self_attn.W_query.weight', 'trf_blocks.0.self_attn.W_key.weight', 'trf_blocks.0.self_attn.W_value.weight', 'trf_blocks.0.self_attn.out_proj.weight', 'trf_blocks.0.self_attn.out_proj.bias', 'trf_blocks.0.ffn.fc1.weight', 'trf_blocks.0.ffn.fc1.bias', 'trf_blocks.0.ffn.fc2.weight', 'trf_blocks.0.ffn.fc2.bias', 'trf_blocks.0.norm1.weight', 'trf_blocks.0.norm2.weight', 'trf_blocks.1.self_attn.mask', 'trf_blocks.1.self_attn.W_query.weight', 'trf_blocks.1.self_attn.W_key.weight', 'trf_blocks.1.self_attn.W_value.weight', 'trf_blocks.1.self_attn.out_proj.weight', 'trf_blocks.1.self_attn.out_proj.bias', 'trf_blocks.1.ffn.fc1.weight', 'trf_blocks.1.ffn.fc1.bias', 'trf_blocks.1.ffn.fc2.weight', 'trf_blocks.1.ffn.fc2.bias', 'trf_blocks.1.norm1.weight', 'trf_blocks.1.norm2.weight', 'trf_blocks.2.self_attn.mask', 'trf_blocks.2.self_attn.W_query.weight', 'trf_blocks.2.self_attn.W_key.weight', 'trf_blocks.2.self_attn.W_v

In [49]:
# Load model and checkpoint

print(checkpoint_path)
raw = torch.load(checkpoint_path, map_location='cpu')
if isinstance(raw, dict) and raw.get("model_state_dict") is not None:
    state_dict = raw["model_state_dict"]
    print("Loaded nested model_state_dict from checkpoint")
elif isinstance(raw, dict) and raw.get("state_dict") is not None:
    state_dict = raw["state_dict"]
    print("Loaded nested state_dict from checkpoint")
else:
    state_dict = raw

# Backward-compat: fix typos/prefixes sometimes present in checkpoints
if 'embbeding.token_embeddings.weight' in state_dict:
    corrected = {}
    for k, v in state_dict.items():
        corrected[k.replace('embbeding.', 'embedding.')] = v
    state_dict = corrected

for prefix in ("_orig_mod.", "module."):
    if any(k.startswith(prefix) for k in state_dict):
        state_dict = {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in state_dict.items()}

print("state_dict sample keys:", list(state_dict.keys())[:10])

# Helper to fetch tensors by suffix for robustness
def get_tensor_by_suffix(suffix: str):
    for k, v in state_dict.items():
        if k.endswith(suffix):
            return k, v
    return None, None

# Validate vocab size compatibility using embedding or output head weights
_, emb_tensor = get_tensor_by_suffix("embedding.token_embeddings.weight")
if emb_tensor is None:
    _, emb_tensor = get_tensor_by_suffix("out_head.weight")
if emb_tensor is None:
    raise KeyError("No embedding or output head weight found in checkpoint; first keys: " + str(list(state_dict.keys())[:20]))

original_vocab_size = emb_tensor.shape[0]
if config["vocab_size"] is None:
    config["vocab_size"] = original_vocab_size
if original_vocab_size != config['vocab_size']:
    raise ValueError(f"Vocabulary size mismatch: checkpoint={original_vocab_size}, expected={config['vocab_size']}")

# Infer FFN hidden dimension from checkpoint to align with saved weights
_, fc2_weight = get_tensor_by_suffix("ffn.fc2.weight")
if fc2_weight is not None:
    ckpt_hidden_dim = fc2_weight.shape[1]
    desired_hidden = config["ffn_hidden_dim"] or int(round(config.get("ffn_expansion", 8/3) * config['emb_dim']))
    if ckpt_hidden_dim != desired_hidden:
        print(f"Adjusting FFN hidden dim to checkpoint value: checkpoint={ckpt_hidden_dim}, expected={desired_hidden}")
        config["ffn_hidden_dim"] = ckpt_hidden_dim

# Create model after finalizing config
model = gpt.GPTModel(config)

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Loaded state_dict. Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")

# Move model to device
model = model.to(device)
model.eval()

print("Model loaded successfully!")


models/full/model_epoch_1.pt
Loaded nested model_state_dict from checkpoint
state_dict sample keys: ['embedding.token_embeddings.weight', 'trf_blocks.0.self_attn.mask', 'trf_blocks.0.self_attn.W_query.weight', 'trf_blocks.0.self_attn.W_key.weight', 'trf_blocks.0.self_attn.W_value.weight', 'trf_blocks.0.self_attn.out_proj.weight', 'trf_blocks.0.self_attn.out_proj.bias', 'trf_blocks.0.ffn.fc1.weight', 'trf_blocks.0.ffn.fc1.bias', 'trf_blocks.0.ffn.fc2.weight']
Adjusting FFN hidden dim to checkpoint value: checkpoint=683, expected=1365
Loaded state_dict. Missing keys: 0, Unexpected keys: 0
Model loaded successfully!


In [43]:
# Simple generation helper using repo's functions

def generate(prompt: str, max_new_tokens: int = None, temperature: float = None):
    max_tokens = max_new_tokens if max_new_tokens is not None else globals().get('max_new_tokens', 128)
    temp = temperature if temperature is not None else globals().get('temperature', 1.0)
    with torch.no_grad():
        model_device = next(model.parameters()).device
        encoded = tokenizer.encode(prompt)
        input_ids = torch.tensor(encoded, dtype=torch.long, device=model_device).unsqueeze(0)
        out_ids = gpt.generate_new_tokens(
            model=model,
            idx=input_ids,
            max_new_tokens=max_tokens,
            context_size=config['context_length'],
            temperature=temp,
        )
        # Move to CPU for decoding only
        out_ids = out_ids.to('cpu')
        return tokenizer.decode(out_ids.squeeze(0).tolist())

# Quick smoke test
print(generate("What is the capital of the moon?", max_new_tokens=32, temperature=0.2))


What is the capital of the moon?................................


## Chat-style prompting (optional)
You can format prompts with the special tokens used in training for chat-like behavior. Example:

```
<|system|>You are a helpful assistant.<|end|>
<|user|>Write a haiku about autumn leaves.<|end|>
<|assistant|>
```

Then call `generate(prompt)`; the model will continue the assistant response.


In [31]:
import torch

def _top_k_top_p_filter(logits, top_k=50, top_p=0.9):
    if top_k and top_k > 0:
        kth, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
        cutoff = kth[..., -1, None]
        logits = torch.where(logits < cutoff, torch.full_like(logits, float('-inf')), logits)
    if top_p and 0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        csum = torch.cumsum(probs, dim=-1)
        mask = csum > top_p
        mask[..., 0] = False
        sorted_logits[mask] = float('-inf')
        logits.scatter_(-1, sorted_idx, sorted_logits)
    return logits

def generate_plain(prompt: str,
                   max_new_tokens=50,
                   temperature=0.7,
                   top_k=50,
                   top_p=0.9,
                   repetition_penalty=1.2,
                   no_repeat_ngram_size=3):
    device = next(model.parameters()).device
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = ids[:, -config['context_length']:]
            logits = model(idx_cond)[:, -1, :]

            # repetition penalty (simple)
            if repetition_penalty and repetition_penalty != 1.0:
                seen = ids[0].unique()
                logits[0, seen] /= repetition_penalty

            # no-repeat n-grams
            if no_repeat_ngram_size and ids.size(1) >= no_repeat_ngram_size:
                n = no_repeat_ngram_size
                recent = ids[0, -n+1:].tolist()
                if len(recent) == n-1:
                    # find all tokens that would complete a repeated n-gram
                    for i in range(ids.size(1)-n+1):
                        if ids[0, i:i+n-1].tolist() == recent:
                            ban_tok = ids[0, i+n-1].item()
                            logits[0, ban_tok] = float('-inf')

            logits = logits / max(1e-6, temperature)
            logits = _top_k_top_p_filter(logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids.squeeze(0).to('cpu').tolist())

In [37]:
print(generate_plain("your name is",
                     max_new_tokens=40,
                     temperature=0.2,
                     top_k=20,
                     top_p=0.8,
                     repetition_penalty=1.2,
                     no_repeat_ngram_size=3))

your name is...g..ar..ances..é..-..ric..in..ung..ural..va..ula..r.. of


prompt = """<|system|>You are a friendly tutor.<|end|>
<|user|>Explain why leaves change color in autumn.<|end|>
<|assistant|>"""
print(generate(prompt, max_new_tokens=80, temperature=0.2))

In [39]:
prompt = """<|system|>You are a friendly tutor.<|end|>
<|user|>Explain why leaves change color in autumn.<|end|>
<|assistant|>"""
print(generate_plain(prompt,
                     max_new_tokens=80,
                     temperature=0.2,
                     top_k=20,
                     top_p=0.8))

<|system|>You are a friendly tutor.<|end|>
<|user|>Explain why leaves change color in autumn.<|end|>
<|assistant|>ororor...ararar..va..ric..ic..in..g..éararas..---..ural..ances..ung..v..t.. '..cc..ulararororararic.ar.arula..at..r..ley


In [44]:
prompt = """MOON"""
print(generate_plain(prompt,
                     max_new_tokens=80,
                     temperature=0.2,
                     top_k=20,
                     top_p=0.8))

MOON...g..ararar..ricricricororor..---.. '..é..ural..ances..in..ung..at..r.. of..ic..va..ory..cc..v..am..ens..ley..as..t.
